[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/06_ai_engineering/21_llm_fundamentals.ipynb)

# 📓 Notebook 21 — LLM Fundamentals

> **Module:** AI Engineering · **Estimated time:** ~75 minutes · **Difficulty:** Intermediate

> 📍 **Concepts are prerequisite-free.** You can read this early (alongside `00c`) to demystify *what an LLM is*; the light maths in §4 and §6 lands more easily once you've seen vectors in NB 23, so skim those two sections on a first pass if needed.

Module 6 opens with the question every learner asks first: *what is an LLM actually doing inside?* You will leave this notebook able to draw the Transformer, explain the Attention mechanism, distinguish pre-training from fine-tuning from prompting, and use the right prompting technique for the right task.

This is the *theory* notebook. NB 22–26 are the hands-on ones.

> 🏢 **Our running example — meet "Helpa."** Imagine you've just been handed a project: build **Helpa**, an AI customer-support assistant for a small SaaS company. Throughout this notebook we'll keep returning to Helpa to ground each idea — *what does this concept mean for the assistant we're trying to ship?* Tokens become Helpa's monthly bill; attention becomes how Helpa reads a customer's question; hallucinations become the support reply that invents a product that doesn't exist. The theory is the same either way — but it's far easier to remember when it's attached to something you're building.

> 🧭 **The one mental model to keep.** Before any maths, hold this picture: **an LLM is a next-token predictor run in a loop.** Generating text is just *sample → append → repeat*. Everything else in this notebook — size, attention, training, prompting — exists to make that single next-token guess good. We'll call this back at every turn ("remember the *loop* mental model — here it is again").

---

## 🎯 Learning objectives

1. **Explain** what a Large Language Model is and how it generates text, one token at a time.
2. **Distinguish** tokens, parameters, and the training objective (next-token prediction with cross-entropy loss).
3. **Describe** the Transformer architecture and the Attention mechanism (Q/K/V) in your own words.
4. **Distinguish** pre-training, fine-tuning, and prompting — three different intervention points.
5. **Name** five prompting techniques and pick the right one for a given task.
6. **Identify** the two structural limitations (hallucinations, knowledge cutoff) and explain why they exist.

**Prerequisites:** None for this notebook specifically.

**Time budget:** ~75 minutes including exercises.


## 1. What is a Large Language Model?

Before we wire Helpa up to anything, we need to know what kind of thing it *is* — because that one fact explains everything Helpa can and can't do.

An **LLM** is a *neural network* — a mathematical function with billions of tunable numbers called **parameters** — trained on huge volumes of text. Mechanically it works like a powerful autocomplete: given an input, it predicts the *most likely next piece of text*, one small unit called a **token** at a time.

The core idea in three bullets:

- LLMs learn statistical relationships between words from massive text corpora (Wikipedia, books, code, web).
- When asked a question, the model computes the most probable continuation as a response.
- Useful analogy: imagine someone who has read every book ever written. They can continue any topic plausibly — but they have *no idea* what they read yesterday vs. ten years ago.

**Well-known examples in 2026:**

ChatGPT (OpenAI), Claude (Anthropic), Gemini (Google), Llama (Meta), Mistral, DeepSeek.

> 🧭 **Remember the loop mental model.** When a customer types *"How do I reset my password?"*, Helpa doesn't "look up the answer" — it predicts a plausible next token, appends it, and repeats until the reply is done. That single fact is the seed of both its magic (it can phrase anything) and its risks (it can phrase a *wrong* thing just as fluently). The next two sections unpack the two ingredients of that prediction: **tokens** (what it reads) and **parameters** (what it knows).


## 2. How LLMs see text: tokens

Why care about tokens? Because the moment Helpa goes live, *tokens are the unit on your invoice and the limit on how much a customer message can contain*. Get the token picture wrong and you either overspend or truncate someone's question mid-sentence.

A **token** is the smallest unit of text the model processes — typically a short sub-word of 3–4 characters. Models do not see raw characters or whole words; they see sequences of integer **token IDs**.

| Input text | Tokens (one common tokenizer) |
|---|---|
| `ChatGPT` | `[Chat][G][PT]` — 3 tokens |
| `Bielefeld` | `[Bi][ele][feld]` — 3 tokens |
| `The cat sat.` | `[The][_cat][_sat][.]` — 4 tokens |
| `?? !!` | `[?][?][_!][!]` — 4 tokens |

**Why tokens matter in practice:**

- **Cost and limits.** API pricing and the *context window* (e.g. 200,000 tokens) are measured in tokens — not characters, not words.
- **Rule of thumb.** 1 English token ≈ 4 characters ≈ 0.75 words. So 1,000 tokens ≈ 750 English words ≈ 4 paragraphs.
- **Non-English is more expensive.** German, French, Chinese, and especially programming code tokenize less efficiently — more tokens for the same information, higher cost.
- **The model is blind to characters within a token.** Asking *"how many r's are in strawberry?"* is surprisingly hard for LLMs precisely because they don't see characters — they see token IDs.

> 🏢 **Helpa's bill.** Every customer message, every snippet of help-centre context you paste in, and every word Helpa replies all count as tokens — and you pay for both directions. The loop mental model makes this concrete: each *append* step in *sample → append → repeat* is another token you're billed for. Practice exercise 1 puts real numbers on a quarter of support tickets.


In [ ]:
# Tiny demonstration of tokenization (no API call — just shows the idea).
# In real usage you would use tiktoken (OpenAI) or the tokenizer of your specific model.

def naive_token_count(text: str) -> int:
    """Very rough estimate using the 4-chars-per-token rule of thumb."""
    return max(1, len(text) // 4)

samples = [
    'Hello world',
    'Bielefeld is a city in Germany.',
    'The quick brown fox jumps over the lazy dog.',
    'def factorial(n): return 1 if n <= 1 else n * factorial(n - 1)',
]
for s in samples:
    print(f'{naive_token_count(s):>3} tokens (approx)  |  {len(s):>3} chars  |  {s}')


---

### ✋ Quick exercise (~2 min) — Size a Helpa message

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

A customer pastes a 1,200-character message into Helpa. Using the 4-chars-per-token rule of thumb from this section, estimate how many tokens it is, then decide whether it fits inside a 200,000-token context window.

```python
customer_message_chars = 1200
context_window = 200_000
```

In [ ]:
# ✍️ Your turn 👇
customer_message_chars = 1200
context_window = 200_000

# Estimate the tokens (4-chars-per-token rule), then check whether it fits:


<details>
<summary>✅ <b>Solution</b></summary>

```python
est_tokens = customer_message_chars // 4   # ≈ 300 tokens
fits = est_tokens < context_window
print(est_tokens, fits)   # 300 True
```

A 1,200-character message is roughly 300 tokens (4 chars ≈ 1 token) — a tiny fraction of a 200,000-token window, so it fits with room to spare for Helpa's reply.
</details>

## 3. What are parameters? Why "large"?

Earlier we said tokens are what the model *reads*; parameters are what it *knows*. For Helpa this is the first real product decision you'll face: a small model is cheap and fast but blander, a large one is sharper but pricier — and "how big?" is a question about parameters.

**Parameters** are the numerical weights inside the neural network — think of them as billions of tiny tuning knobs that get adjusted during training. After training, **these numbers *are* the model's knowledge**.

| Model | Parameters | Year |
|---|---|---|
| GPT-2 | 1.5 billion | 2019 |
| Llama-3 (open-weights) | 8B / 70B / 405B | 2024 |
| GPT-4, Claude, Gemini (frontier) | hundreds of billions (estimated) | 2023+ |

**Why "large"?**

More parameters = greater capacity to absorb patterns from training data = usually better quality, but also more compute, more memory, more latency, and more cost.

> ⚠️ **Key insight.** There is no manual list of facts inside the model. Knowledge is *implicitly distributed* across all the weights, which is why you cannot "edit" a single fact in an LLM the way you would update a row in a database. This is exactly why **RAG** exists (NB 29) — to attach a real database to the model.

> 🏢 **Helpa's dilemma, foreshadowed.** Because your SaaS product names and pricing live *nowhere* in those frozen weights, no amount of "large" will let Helpa answer *"what's in the Pro plan?"* reliably. Hold that thought — it's the seed of §10's knowledge-cutoff limitation and the whole reason RAG exists. (Practice exercise 2 walks through picking a model size for Helpa-style scenarios.)


## 4. 🧠 The training objective: next-token prediction

We've described *what* an LLM does (predict tokens) and *what it's made of* (parameters). The natural next question — and the one that demystifies the whole field — is *how does it ever get good at that?* The answer is a single, almost embarrassingly simple training goal.

In plain words: **hide the next token in a stretch of text, ask the model to predict it, measure the error, nudge every parameter slightly toward a better prediction. Repeat trillions of times.**

| | |
|---|---|
| **Visible context** | `The capital of France is` |
| **True next token** | `_Paris` |
| **Model's guess** | a probability over *all* tokens; `Paris` should be highest |
| **Loss** | *cross-entropy*: how surprised was the model by the true token? |

**Why this single goal is enough.** To predict the next token well across billions of contexts, the model implicitly has to learn grammar, world facts, code syntax, multi-step reasoning — every one of these reduces prediction error. *The behaviours that look like "understanding" are side-effects of getting really good at one narrow task.*

Cross-entropy loss in a single line:

$$\mathcal{L} = - \sum_{i} y_i \log \hat{p}_i$$

where $y_i = 1$ for the true next token and $0$ for all others (a so-called *one-hot* target), and $\hat{p}_i$ is the model's predicted probability for token $i$. Lower loss = the model is less surprised = better predictions.


### 🔬 What actually happens when an LLM "generates text"

Section 4 said the model predicts *the next token*. But how does that single prediction turn into a whole paragraph? The secret is that **generation is one tiny step run in a loop**. Let's open the box.

There are two mechanics, chained together:

**(1) Tokenization — the model never sees your text.** Before anything happens, your string is chopped into **tokens** (sub-word pieces) and each token is looked up in a fixed **vocabulary** to get an integer **id**. The model only ever sees those ids — little integers, not letters.

```text
   "the cat"
        │  tokenizer splits + looks up ids
        ▼
   ["the", " cat"]   →   [5, 8]        ← the model sees ONLY [5, 8]
```

**(2) Next-token prediction — one step.** Given the ids so far (the *context*), the model outputs one number per vocabulary entry — the raw **logits**. Turn those logits into probabilities, and you have a full probability distribution over *every possible next token*:

```text
   context [5, 8]
        │  model forward pass
        ▼
   logits  = one raw score per vocab token   →  [2.1, -0.4, 3.0, ...]
        │  softmax
        ▼
   probs   = a probability per vocab token    →  [0.18, 0.02, 0.55, ...]   (sums to 1.0)
```

**The generate loop.** You *pick* one token from that distribution, **append** it to the context, and **repeat** — feeding the now-longer context back in. That sample-append-repeat loop is what "autoregressive generation" means:

```text
   ┌───────────────────────────────────────────────────────────┐
   │                                                           │
   ▼                                                           │
context ──► model ──► logits ──► softmax ──► probs ──► PICK one │
  [ids]                                                  │     │
   ▲                                                     │     │
   │                          append the picked id ──────┘     │
   └───────────────────────────────────────────────────────────┘
            stop when you hit an end token or a length limit
```

> 🧠 **One sentence to keep.** *An LLM is a next-token predictor run in a loop; "generating text" is **sample → append → repeat**.* Everything else (size, attention, training) is in service of making that one next-token guess good.


### Greedy vs sampling — how you "PICK one"

The model hands you a probability distribution; *picking* a token from it is a separate decision **you** control. Two basic strategies:

| Strategy | How it picks | Behaviour |
|---|---|---|
| **Greedy** (`argmax`) | always take the single highest-probability token | deterministic — same input ⇒ same output, can feel flat/repetitive |
| **Sampling** | draw a token at random, weighted by the probabilities | varied — same input ⇒ different outputs run to run |

Greedy is "play it safe, always the favourite". Sampling is "roll a weighted die" — the most likely token usually wins, but less likely ones sometimes get their turn, which is what makes output feel creative and non-robotic.

The next cell builds a **tiny offline toy LLM** — a 6-word vocab and a *fake* `next_token_logits(context)` (just a lookup table, no neural net) — so we can watch the whole loop with real numbers. No model, no API, fully seeded.


In [ ]:
import numpy as np

rng = np.random.default_rng(42)   # seed → reproducible "randomness"

# ── A tiny inline vocabulary: token string  <->  integer id ──
VOCAB = ["the", "cat", "sat", "on", "mat", "<end>"]
stoi  = {tok: i for i, tok in enumerate(VOCAB)}   # string -> id
itos  = {i: tok for i, tok in enumerate(VOCAB)}   # id -> string

def encode(tokens):                 # tokenizer:  ["the","cat"] -> [0, 1]
    return [stoi[t] for t in tokens]

def decode(ids):                    # de-tokenizer: [0, 1] -> "the cat"
    return " ".join(itos[i] for i in ids)

print("VOCAB        :", VOCAB)
print("encode(...)  :", encode(["the", "cat"]), " <- the model sees ONLY these ids")
print("decode(...)  :", decode([0, 1]))

# ── A FAKE 'model': map the last token -> raw logits over the whole vocab. ──
# A real LLM computes these with billions of parameters from the FULL context.
# Ours just looks at the previous token. Higher logit = model likes that token more.
#                  the   cat   sat   on    mat   <end>
LOGIT_TABLE = {
    "the":   [0.0,  4.0,  0.5,  0.2,  1.0, -3.0],   # after "the" -> usually "cat"
    "cat":   [0.0,  0.0,  4.0,  0.3,  0.2, -3.0],   # after "cat" -> usually "sat"
    "sat":   [0.0,  0.0,  0.0,  4.0,  0.5, -2.0],   # after "sat" -> usually "on"
    "on":    [0.0,  0.0,  0.0,  0.0,  4.0, -2.0],   # after "on"  -> usually "mat"
    "mat":   [-2.0, -2.0, -2.0, -2.0, 0.0,  4.0],   # after "mat" -> usually "<end>"
    "<end>": [0.0,  0.0,  0.0,  0.0,  0.0,  9.0],   # stays ended
}

def next_token_logits(context_ids):
    """Toy stand-in for an LLM forward pass: ids -> one raw logit per vocab token."""
    last_tok = itos[context_ids[-1]]            # this toy only reads the last token
    return np.array(LOGIT_TABLE[last_tok], dtype=float)

# Peek at the raw logits the 'model' produces after the token "the":
logits = next_token_logits(encode(["the"]))
print("\nlogits after 'the':", logits, " <- one raw score per vocab token")


### Softmax with temperature — the randomness dial

Raw logits aren't probabilities (they can be negative, and don't sum to 1). **Softmax** converts them: exponentiate each, then divide by the total so they sum to 1.

The twist is **temperature `T`**: before softmax, divide every logit by `T`. That one division reshapes the whole distribution:

$$ p_i = \frac{e^{\,z_i / T}}{\sum_j e^{\,z_j / T}} $$

- **Low `T` (e.g. 0.2)** → big logits get exaggerated → the distribution gets **sharp/peaky** → near-deterministic (close to greedy).
- **`T = 1.0`** → the model's distribution, untouched.
- **High `T` (e.g. 2.0)** → differences get squashed → the distribution gets **flat** → more random, more "surprising" tokens.

```text
        same logits, three temperatures
   T=0.2  ▏▁▁█▁▁    sharp  → almost always the favourite (deterministic)
   T=1.0  ▏▂▃█▅▂    medium → favourite usually wins, others get a turn
   T=2.0  ▏▄▅█▆▄    flat   → near-uniform, anything can happen
```

> 🎯 **Mental model.** Temperature is the **randomness dial**. Turn it *down* for factual, repeatable answers (extraction, classification); turn it *up* for brainstorming and variety. It changes *only how you pick*, never what the model knows.

Now let's implement it and watch the same logits reshape at `T = 0.2`, `1.0`, and `2.0`.


In [ ]:
def softmax(logits, temperature=1.0):
    """Convert raw logits -> probabilities, reshaped by temperature T."""
    z = np.asarray(logits, dtype=float) / temperature
    z = z - z.max()                      # subtract max for numerical stability
    e = np.exp(z)
    return e / e.sum()

# The SAME logits (after the token "the"), viewed at three temperatures:
logits = next_token_logits(encode(["the"]))

print("token   :", "  ".join(f"{t:>5}" for t in VOCAB))
for T in (0.2, 1.0, 2.0):
    p = softmax(logits, temperature=T)
    bars = "  ".join(f"{x:5.2f}" for x in p)
    label = {0.2: "sharp", 1.0: "medium", 2.0: "flat"}[T]
    print(f"T={T:<4} :  {bars}   ({label}; sums to {p.sum():.2f})")

# Notice: at T=0.2 almost all probability piles onto 'cat' (the top logit);
# at T=2.0 the probabilities spread out and the long-shots get a real chance.


In [ ]:
def generate(prompt_tokens, strategy="greedy", temperature=1.0, max_new=8, seed=42):
    """The autoregressive loop: context -> logits -> pick -> append -> repeat."""
    local_rng = np.random.default_rng(seed)
    ids = encode(prompt_tokens)                 # tokenize the prompt
    for _ in range(max_new):
        logits = next_token_logits(ids)         # 1. model: context -> logits
        probs  = softmax(logits, temperature)   # 2. softmax (with temperature)
        if strategy == "greedy":
            nxt = int(np.argmax(probs))         # 3a. PICK: always the favourite
        else:                                   # strategy == "sample"
            nxt = int(local_rng.choice(len(VOCAB), p=probs))  # 3b. weighted die
        ids.append(nxt)                         # 4. append, then loop
        if itos[nxt] == "<end>":                # stop on the end token
            break
    return decode(ids)

prompt = ["the"]

# ── Greedy: deterministic — run it twice, identical output ──
print("greedy   #1 :", generate(prompt, strategy="greedy"))
print("greedy   #2 :", generate(prompt, strategy="greedy"))

# ── Sampling at T=1.0: varied — different seeds give different continuations ──
print("sample s4   :", generate(prompt, strategy="sample", temperature=1.0, seed=4))
print("sample s8   :", generate(prompt, strategy="sample", temperature=1.0, seed=8))

# ── Temperature's effect on sampling, same seed ──
print("sample T=0.2:", generate(prompt, strategy="sample", temperature=0.2, seed=3),
      " (cold: hugs the greedy path)")
print("sample T=2.0:", generate(prompt, strategy="sample", temperature=2.0, seed=3),
      " (hot: wanders off)")


That short loop *is* the whole show. Read it back against the diagram:

1. **Tokenize** the prompt to ids — the only thing the model ever sees.
2. **`next_token_logits`** → one raw score per vocab token (a real LLM uses billions of parameters and the full context here; our toy just reads the last token).
3. **`softmax(..., temperature)`** → a probability distribution, reshaped by the randomness dial.
4. **Pick** — `argmax` (greedy) or a weighted draw (sampling).
5. **Append** the picked id and **loop**, stopping at `<end>` or a length cap.

Notice the proof in the output: **greedy ran twice gives the identical sentence** (deterministic), while **sampling with different seeds wanders down different paths** — and **cold `T=0.2` hugs the greedy path** while **hot `T=2.0` strays**. Same toy model, same logits: the *only* thing you changed was *how you pick*.

> 🧠 **Tie it back to the real thing.** Swap our 6-word `LOGIT_TABLE` for a Transformer that produces logits over a ~100,000-token vocabulary from the full context, and you have a real LLM. The generate loop, the softmax, the temperature dial, greedy-vs-sample — all of it is *exactly* what you implemented above. The 405-billion-parameter version differs only in how good step 2's guess is.

> ⚠️ **Common confusion.** Temperature does **not** make the model "smarter" or change its knowledge — it only reshapes the pick. And `temperature=0` isn't really `softmax(0)` (division by zero); libraries treat it as a special case meaning *pure greedy*. For repeatable, factual outputs use a low temperature; for variety, raise it.


---

### ✋ Quick exercise (~2 min) — Predict the greedy path

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Without running anything, trace the toy model from the prompt `["the"]` using **greedy** decoding — at each step pick the highest logit in `LOGIT_TABLE`. What full sentence does it produce, and why is it the same every run? Then confirm with `generate`.

In [ ]:
# ✍️ Your turn 👇
# Trace the greedy path by hand first (read LOGIT_TABLE), then check yourself:
# print(generate(["the"], strategy="greedy"))


<details>
<summary>✅ <b>Solution</b></summary>

```python
print(generate(["the"], strategy="greedy"))
# the cat sat on mat <end>
```

Greedy always walks the chain of highest logits in `LOGIT_TABLE`: the → cat → sat → on → mat → `<end>`. Because it never rolls the die, this deterministic path is the same on every run — sampling is what would let it diverge.
</details>

## 5. The architecture: the Transformer

So far we've treated the box that turns context into logits as a black box (in the toy code it was just a lookup table). Now let's name what's inside that box in a real model — because the answer is the single invention that made Helpa possible at all.

The **Transformer** (Vaswani et al., 2017, *Attention Is All You Need*) is the architecture that made modern LLMs possible. It replaced earlier sequential designs (RNN, LSTM) with a parallelisable structure that scales to billions of parameters.

**The core innovation: the Attention mechanism.** Each token dynamically "listens to" every other token in the context, so meaning is computed from the *full* surrounding text — not just the previous word.

```
  Old design (RNN, sequential)            New design (Transformer, parallel)
  ───────────────────────────             ───────────────────────────────
  word_1 → word_2 → word_3 → ...           word_1 ─┐
  one step at a time,                       word_2 ─┼─► every word can
  cannot parallelise.                       word_3 ─┤   attend to every
                                             ...    ┘   other word.
```

This single change is what made GPUs efficient at training language models, which is what made LLMs scale to billions of parameters, which is what made the 2020s AI wave possible.


## 6. 🧠 The Attention mechanism — a closer look

Here's why this matters for Helpa. A customer writes: *"It stopped working after the update — can you fix it?"* To answer, Helpa has to figure out what *"it"* refers to and which *"update"* — and the only way is to weigh every other word in the message against that one. That weighing *is* attention.

For *every* token, the model produces three learned vectors: a **Query (Q)**, a **Key (K)**, and a **Value (V)**. A token's Query is matched against every other token's Key; the softmax over the scores yields *weights* that say "how much should I listen to each other token?". The token's new representation is then the *weighted sum of all Values*.

| Step | What happens |
|---|---|
| 1. Project | Compute Q, K, V for every token via learned matrices. |
| 2. Score | $\mathrm{score}_{ij} = Q_i \cdot K_j / \sqrt{d_k}$ |
| 3. Normalise | Softmax over $j$ → weights summing to 1 |
| 4. Mix | New vector = $\sum_j \mathrm{weight}_{ij} \cdot V_j$ |

> 💡 $d_k$ is the **length of each Key/Query vector**; dividing by $\sqrt{d_k}$ keeps the dot-product scores from blowing up (and the softmax from saturating) when the vectors are long.

**Why this design wins:**

- **Long-range.** Position 1 directly attends to position 1000 — no information bottleneck.
- **Parallel.** All tokens are processed at once during training (huge speedup vs RNN).
- **Multi-head.** Multiple parallel attention heads capture different kinds of relations (subject-verb, modifier, coreference, code structure, etc.).

> 🎯 **Intuition.** Imagine reading a sentence and, for *each* word, looking at every other word in the sentence and deciding how much to weight it. That's attention. The model does this with learned matrices, in parallel, across many heads, for every layer of the network.

> 🧭 **Remember the loop mental model.** Attention is *inside* step 2 of the loop — it's how the "model forward pass" turns the whole context into the logits we then softmax-and-pick. Better attention ⇒ a better next-token guess ⇒ Helpa understanding *"it"* correctly. (Practice exercise 3 has you explain attention to a stakeholder, and stretch D walks through the famous *"it"* example in detail.)


---

### ✋ Quick exercise (~2 min) — Compute attention weights

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

A token's Query `q` is compared against three Keys. Apply the attention recipe from the table above — score each Key with `q · k / sqrt(d_k)`, then softmax the scores into weights. Which Key does the token attend to most? (Reuse the `softmax` you defined in §4.)

```python
import numpy as np
q  = np.array([1.0, 0.0, 1.0])
k1 = np.array([1.0, 0.0, 1.0])   # "animal"
k2 = np.array([0.0, 1.0, 0.0])   # "street"
k3 = np.array([1.0, 0.0, 0.5])   # "tired"
```

In [ ]:
# ✍️ Your turn 👇
import numpy as np
q  = np.array([1.0, 0.0, 1.0])
k1 = np.array([1.0, 0.0, 1.0])   # key for "animal"
k2 = np.array([0.0, 1.0, 0.0])   # key for "street"
k3 = np.array([1.0, 0.0, 0.5])   # key for "tired"

# Compute the scores (q · k / sqrt(d_k)), softmax them, and see which key wins:


<details>
<summary>✅ <b>Solution</b></summary>

```python
d_k = q.shape[0]
scores = np.array([q @ k1, q @ k2, q @ k3]) / np.sqrt(d_k)
weights = softmax(scores)        # softmax was defined back in §4
print(weights.round(3))          # highest weight lands on k1 ("animal")
```

`k1` is identical to the query, so it has the largest dot product and wins the most attention after the softmax — exactly the Q·K matching → normalise → weighted mix from the §6 table. (`softmax` is reused from the toy-LLM code in §4.)
</details>

## 7. Pre-training, fine-tuning, prompting — three intervention points

We now know how a model is built and how it reads. The practical question for *you*, Helpa's builder, is: **where can you actually influence it?** You won't touch the billions of weights yourself — but you have three very different levers, and choosing the wrong one wastes weeks.

An LLM passes through up to three phases. Knowing which one to use for a given problem saves a lot of wasted effort.

**Phase 1 — Pre-training.** Done *once* by the model provider on massive datasets. Adjusts billions of parameters via next-token prediction. Extremely compute-intensive (millions of dollars, weeks on thousands of GPUs). *You as a user never do this.*

**Phase 1.5 — Fine-tuning** *(optional).* Further train the model on a much smaller, curated dataset to specialise its behaviour. Hours-to-days on far fewer GPUs.

| Fine-tuning flavour | What it teaches the model |
|---|---|
| **Supervised Fine-Tuning (SFT)** | Imitate good demonstrations: tone, format, domain phrasing. |
| **Instruction tuning** | Reliably follow user instructions across many task types. |
| **RLHF / preference tuning** | Prefer outputs humans rate as helpful, harmless, honest. |
| **Parameter-efficient (LoRA)** | Train a small adapter only — cheap and reversible. |

**Phase 2 — Prompting.** What you do every time you use a chatbot or call an API. The prompt steers the already-trained model toward the desired task. **The art of precise formulation is called prompt engineering.**

> 💡 **Fine-tuning vs RAG.** Fine-tuning changes the model's *weights* — good for changing *behaviour or style*. RAG (NB 29) keeps weights frozen and supplies external *facts* at query time — good for changing *knowledge*. Pick fine-tuning for *how* the model speaks, pick RAG for *what* the model knows.

> 🏢 **For Helpa, the lever is prompting (with RAG later).** You want Helpa to *sound* on-brand and *answer from* your help-centre. The first is a prompting/fine-tuning question; the second is a RAG question. For a small SaaS company, prompting + RAG gets you 95% of the way before fine-tuning is ever worth the cost — which is exactly why the next section is about prompting.


## 8. Common prompting techniques

Prompting is the lever you'll pull most, so it pays to know more than one way to pull it. Each technique below is a different way of *shaping the context* before the loop runs — and the right one depends entirely on the task you're handing Helpa.

| Technique | Prompt example | Typical result |
|---|---|---|
| **Zero-shot** | *"What is a SWOT analysis?"* | General, textbook-style explanation. |
| **Few-shot** | *"Classify: 'Great product!' → Positive. 'Broken.' → Negative. 'Damaged.' →"* | Consistent classification in the given format. |
| **Chain-of-Thought** | *"Analyse step by step whether a 10% price rise is advisable."* | Structured analysis with traceable reasoning. |
| **Role assignment** | *"You are a senior financial analyst. Review this Q4 report and..."* | Domain-appropriate tone and depth. |
| **Format constraint** | *"Return your answer as a JSON object with keys 'verdict' and 'reasoning'."* | Machine-parseable output. |

**The 5-part anatomy of a good prompt** (here, the exact prompt you'd give Helpa):

1. **Role** — *"You are Helpa, a friendly support agent for our SaaS product."*
2. **Context** — *"The customer is on the Pro plan and is asking how to export their data."*
3. **Task** — *"Draft a 2-paragraph reply with clear steps."*
4. **Format** — *"Use plain text. No greetings or sign-off."*
5. **Constraints** — *"Only describe features that exist. If unsure, say you'll escalate to a human."*

Most weak prompts skip 3 of the 5. Strong prompts hit all 5 in 2-4 sentences. (Notice constraint 5 — *"if unsure, escalate"* — is your first line of defence against the hallucination problem we hit next.)


---

### ✋ Quick exercise (~2 min) — Assemble a 5-part prompt

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Using the 5-part anatomy (Role · Context · Task · Format · Constraints) from the section above, build one prompt string for Helpa to answer a customer asking **how to reset their password**. Fill in all five parts, then join them into a single prompt.

In [ ]:
# ✍️ Your turn 👇
# Fill in all five parts, then join them into one prompt string:
parts = {
    "role":        "...",
    "context":     "...",
    "task":        "...",
    "format":      "...",
    "constraints": "...",
}
# prompt = "\n".join(parts.values())
# print(prompt)


<details>
<summary>✅ <b>Solution</b></summary>

```python
parts = {
    "role":        "You are Helpa, a friendly support agent for our SaaS product.",
    "context":     "The customer is on the Pro plan and forgot their password.",
    "task":        "Write a short reply with clear steps to reset it.",
    "format":      "Use plain text with numbered steps. No greeting or sign-off.",
    "constraints": "Only describe features that exist; if unsure, offer to escalate to a human.",
}
prompt = "\n".join(parts.values())
print(prompt)
```

A strong prompt hits all five parts in a few sentences. Note the final constraint ("if unsure, escalate") — it's the first guardrail against the hallucination problem covered in §9.
</details>

## 9. ⚠️ Limitation 1 — Hallucinations

Everything up to here was about what makes LLMs powerful. The last two sections are the flip side — the two ways Helpa will *let you down* in production, and why both follow directly from the loop mental model.

When an LLM produces output that reads convincingly but is *factually wrong or entirely made up*, we call this a **hallucination**. Because the model optimises for *plausible-sounding* text, it can confidently invent facts, figures, or sources.

**Why this is a real problem for organisations:**

- **Legal risks.** An AI assistant invents a non-existent clause in a contract.
- **Reputational damage.** A chatbot gives a customer incorrect product information.
- **Poor decisions.** A market analysis is based on fabricated statistics.

**Helpa's version of the problem.** A customer asks *"Does the Pro plan include the CloudSync feature?"* Helpa, having no real knowledge of your catalogue, predicts the *plausible* continuation — *"Yes, CloudSync is available on Pro!"* — even though no such feature exists. This is the loop doing exactly its job (pick the likely next token) and being exactly wrong. The reply looks perfectly normal; the only way to know it's false is to check.

> 🧭 **It's the loop, not a bug.** Hallucination isn't Helpa "malfunctioning" — it's *sample → append → repeat* with no grounding. The model is rewarded for plausibility, not truth. That reframing tells you the fix can't be "ask it to try harder"; it has to be *adding real evidence to the context*.

> 🔭 **Mitigation strategies** (covered in detail later):
> - **RAG** (NB 23) — give the model real evidence to ground its answer on.
> - **Tools** (NB 24) — let the model *look things up* instead of recalling.
> - **Evaluation harness** (NB 26) — measure how often it hallucinates so you can catch regressions.


## 10. ⚠️ Limitation 2 — Knowledge cutoff

This is the dilemma we foreshadowed back in §3: your product details live nowhere in the frozen weights. Here's the formal name for it.

An LLM's knowledge is limited to the point in time when its **training was completed**. It has no access to information after that date and no access to *private company data*.

| Limitation | Example |
|---|---|
| Outdated information | The model doesn't know about new products or changed regulations. |
| Lack of timeliness | It cannot make statements about current stock prices or news. |
| No access to internal data | Internal reports, customer databases, emails are unknown. |

**Helpa's version.** Even a perfect, hallucination-free model *still* can't tell a customer that you shipped a new billing tier last Tuesday, or quote *this* customer's current plan — none of it was in the training data, and none of it can be. Knowledge cutoff and "no private data" are two faces of the same wall.

**Core problem.** Standard LLMs cannot access real-time information or private knowledge bases. The solution — again — is **RAG** (NB 29), which decouples *knowledge* from the *model* so the knowledge base can be updated without re-training. (Same fix as hallucinations, for the same reason: when the model can't *know*, give it something to *read*.)


## 11. 🧠 Mental model — the three levels of LLM usage

Both limitations point the same way, so let's zoom out into the map for the rest of Module 6 — and trace where Helpa sits on it.

```
    ┌──────────────────┐    ┌──────────────────┐    ┌──────────────────┐
    │  Just LLM        │    │  LLM + RAG       │    │  LLM + RAG +     │
    │                  │ ─► │                  │ ─► │  Agent           │
    │  Generates text  │    │  Grounded in     │    │  Plans and acts  │
    │  from frozen     │    │  current docs    │    │  via tools       │
    │  weights         │    │                  │    │                  │
    └──────────────────┘    └──────────────────┘    └──────────────────┘
       Hallucination          Precise, verifiable      Autonomous goal
       risk, no fresh         answers with             completion with
       data, no actions       citations                external systems
```

**Where Helpa lives.** A "Just LLM" Helpa is the version that invents CloudSync. Move it one step right — *LLM + RAG* — and it answers from your real help-centre, citing the doc. Move it one step further — *LLM + RAG + Agent* — and it can also *act*: look up the customer's plan, open a ticket, issue a refund. This ladder is Helpa's roadmap, and it's the module's roadmap too.

The rest of Module 6 walks up this ladder: NB 22 is the *hands-on* foundation, NB 23 builds the *RAG* level, NB 24 builds the *Agent* level. And every rung is still the same loop you met in §4 — *sample → append → repeat* — now with better context fed into it.


## 🧪 Practice exercises

### Exercise 1 — ⭐ Token counting and budgeting

Your team needs to feed the previous 90 days of customer-support tickets into an LLM prompt for analysis. You have ~600 tickets averaging 400 English words each.

1. Estimate the total tokens (use the 0.75-words-per-token rule of thumb).
2. Does this fit in a 200,000-token context window? In a 32,000-token window? In a 4,000-token window?
3. If not, name two strategies to make it fit.

<details>
<summary>💡 <b>Solution / Answer</b></summary>

1. **Estimate.** 600 tickets × 400 words ÷ 0.75 words-per-token ≈ **320,000 tokens** total content.

2. **Fit?** Does not fit in 200k (need 320k); fits in *none* of the smaller windows.

3. **Two strategies:**
   - **Summarise first**, then feed the summaries (e.g. one-paragraph summary per ticket → 600 × 50 tokens = 30k tokens — comfortably fits in any window).
   - **Filter / retrieve** the relevant tickets first via embedding similarity (RAG-style, NB 23) — typically pull top-30 most relevant rather than all 600.

*The combined pattern (RAG + per-ticket summary) is the production default for analytics over large unstructured corpora.*
</details>

### Exercise 2 — ⭐⭐ Parameter intuition

Llama-3 ships in three sizes: 8B, 70B, and 405B parameters. Roughly which one would you pick for each scenario, and why?

a. Real-time customer-support chat with strict per-call cost budget.
b. Once-monthly batch summary of all regulatory filings for legal review.
c. Coding assistant running on a developer's laptop (no internet).
d. Internal benchmark to compare against GPT-4 on a hard reasoning task.

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**a. 8B.** Cost-per-call dominates; an 8B model gives plenty of quality for routing and most replies, and is 10–50× cheaper to run than a 70B+ model.

**b. 70B or 405B.** Batch is run once a month — latency and cost matter less; legal review *quality* matters a lot. Use the largest model the budget allows.

**c. 8B.** Memory constrains the choice: a typical laptop GPU has 8–16 GB of VRAM; even quantised, a 70B model is borderline and a 405B impossible. 8B (quantised) runs comfortably.

**d. 405B.** Reasoning benchmarks are exactly where the largest model usually pulls ahead. If you're benchmarking against a frontier model, you want your own *largest* comparable model.

**Key heuristic:** model size choice is usually a *cost × latency × quality* trade-off, not an absolute "bigger is better" decision.
</details>

### Exercise 3 — ⭐⭐ Why attention?

In one paragraph, explain to a non-technical stakeholder why the Transformer's Attention mechanism was such a breakthrough. Use the analogy of reading a sentence to describe what attention does.

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**Example explanation:**

> *"Before the Transformer, language models processed sentences one word at a time, in order — like reading a long paragraph with a single fingertip, where each word's meaning depends on remembering everything you've read so far. This was slow and made long-distance relationships hard: by the time you reached word 100, the context from word 1 had often been blurred or lost.*
>
> *Attention lets the model, for every word, look at every other word in the sentence simultaneously and decide how much each one matters for understanding that word — like glancing at the whole page at once and highlighting the words that are relevant to the one you're thinking about. That single change made two things possible: long-range understanding (the model can connect a pronoun in sentence 5 to its antecedent in sentence 1), and parallel training on GPUs (since every word is processed at the same time, not in sequence). Without attention, today's billion-parameter models would not be trainable in practice."*

**What makes this good:** uses a concrete analogy (the fingertip, the highlighted page), avoids technical jargon, names the two real benefits (long-range + parallelisable), and is one paragraph.
</details>

### Exercise 4 — ⭐⭐ Pick the right technique

For each prompt below, identify which prompting technique it uses (zero-shot, few-shot, chain-of-thought, role assignment, format constraint) and suggest one improvement.

a. *"Classify this review as positive or negative: 'The shipping was slow but the product is great.'"*

b. *"You are an experienced financial analyst. Reviewing the attached Q3 report, identify the three most concerning trends and rank them by severity. Format as a numbered list."*

c. *"Solve this: A train leaves City A at 9am at 60 mph. A second train leaves City B at 10am at 80 mph..."*

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**a. Zero-shot.** No examples, no role, no format. *Improvement:* add 2-3 labelled examples (few-shot) to nail down what counts as positive when there's mixed sentiment.

**b. Role assignment + format constraint.** Combines two techniques. *Improvement:* add chain-of-thought ("think through each trend before ranking") for a more defensible ranking.

**c. Zero-shot.** Just states the problem. *Improvement:* add chain-of-thought explicitly ("Solve this step by step. Show your reasoning before the final answer."). On arithmetic and multi-step reasoning, this single addition typically improves accuracy by 20-40 percentage points on hard problems.
</details>

### Exercise 5 — ⭐⭐ Debug me 🐞

A colleague asks: *"Why does the LLM keep getting our internal product names wrong? It confidently says 'CloudSync' is one of our products, but we don't have a CloudSync product."* In 3-4 sentences, explain what's happening and propose the right fix from this notebook's vocabulary.

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**What's happening:** The model is *hallucinating*. It has no access to your private product catalogue (knowledge-cutoff problem), so when asked about "your products" it fills in the gap with plausible-sounding but fabricated names from training-data patterns (many SaaS companies have a "-Sync" product, so the model produces one).

**The right fix is not prompt-engineering or fine-tuning — it's RAG.** Connect the model to your actual product catalogue (a CSV, an internal API, a vector DB of product descriptions) and instruct the model to answer *only from that data*. NB 29 builds exactly this pattern. Fine-tuning could also encode product names but would be more expensive and harder to update when products change.
</details>

## 🧠 Stretch exercises

### Stretch exercise A — ⭐⭐⭐ Implement next-token prediction

Without using an LLM, write a tiny Python function `naive_next_token(text, corpus)` that, given some text and a corpus of training sentences, returns the *most likely next word* based on simple bigram counts (i.e. for the last word of `text`, find which word most often follows it in `corpus`).

This is a 10-line version of what an LLM does — except the LLM uses billions of parameters and the full context, not a 2-word window.

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**Example implementation:**

```python
from collections import Counter, defaultdict

def naive_next_token(text, corpus):
    # Build a {word -> Counter(next_word -> count)} table from the corpus.
    bigram_counts = defaultdict(Counter)
    for sentence in corpus:
        tokens = sentence.lower().split()
        for i in range(len(tokens) - 1):
            bigram_counts[tokens[i]][tokens[i + 1]] += 1
    
    last_word = text.lower().split()[-1]
    if last_word not in bigram_counts:
        return None  # never seen this word
    return bigram_counts[last_word].most_common(1)[0][0]

corpus = ['the cat sat on the mat', 'the cat ate the food', 'the dog ran fast']
print(naive_next_token('I think the', corpus))  # 'cat' (most common after 'the')
```

**Connecting back to LLMs:** the LLM uses (a) a context window of thousands of tokens instead of just one, (b) learned dense vector representations instead of discrete word counts, and (c) the Attention mechanism to weigh how much each preceding token matters. But the *training objective* — predict the next token — is the same one you just implemented.
</details>

### Stretch exercise B — ⭐⭐⭐ Hallucination experiment

Open your favourite LLM chat interface. Ask:

1. *"Cite three peer-reviewed papers on the effect of caffeine on memory."*
2. *"Cite three peer-reviewed papers on the effect of caffeine on memory. For each, provide the DOI and the year."*
3. *"Cite three peer-reviewed papers on the effect of caffeine on memory. For each, provide the DOI. If you are not certain a paper exists, say so explicitly."*

Then *check whether the cited papers actually exist* (Google Scholar, DOI resolver). Report what you observed across the three prompts.

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**Typical pattern (will vary by model):**

- **Prompt 1:** Model produces three plausible-sounding citations. When you check, *one to three of them do not exist* — invented authors, made-up titles, journals that exist but never published that paper.
- **Prompt 2:** Adding *"DOI and year"* sometimes makes the hallucination *worse* — the model now invents matching DOIs that don't resolve.
- **Prompt 3:** The explicit *"say so if unsure"* clause noticeably reduces fabrication for recent frontier models. The model often hedges: *"I'm not certain about the third one — please verify."*

**Takeaway:** instruction-tuned models can be steered toward calibrated uncertainty *if you ask them to be*. This is exactly the kind of guardrail you should embed in any production prompt that touches factual claims. RAG (NB 29) is a stronger solution, because it forces every citation to come from a real document.
</details>

### Stretch exercise C — ⭐⭐⭐ Explain fine-tuning vs RAG to a stakeholder

A non-technical stakeholder says: *"I keep hearing about fine-tuning AND about RAG. What's the difference? Which one do we need for our project — a chatbot answering customer questions from our 50-page product manual?"*

Write a one-page memo (~200 words) that:
1. Explains the difference in one sentence each.
2. Gives an analogy that makes it intuitive.
3. Recommends the right approach for the project, with one concrete justification.
4. Names one thing they should *not* do.

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**Example memo:**

> **Fine-tuning vs RAG: a one-page decision aid**
>
> *Fine-tuning* permanently changes the AI model's weights based on examples we provide — useful when we want the model to *behave* differently (specific tone, format, domain phrasing). *RAG* (Retrieval-Augmented Generation) keeps the model untouched and instead retrieves relevant chunks from our documents at the moment a question is asked, then asks the model to answer *based on those chunks*.
>
> **Analogy.** Fine-tuning is like sending an employee on a 6-month training course to internalise our domain. RAG is like giving the employee our reference manual and a fast search engine, then asking the question. For *knowledge* that changes (prices, policies, product specs), the manual approach is dramatically more practical.
>
> **Recommendation for our project: RAG, not fine-tuning.** The product manual updates monthly; fine-tuning a new model every month would be expensive and slow. RAG lets us re-index the updated manual in minutes, with the model unchanged. The 50-page size is well within typical RAG capacity.
>
> **What we should *not* do:** treat this as a fine-tuning project. The cost difference is roughly 50–100× for our scale, and the model would still be stale the day after we shipped.

**What makes this memo good:** explicit definitions, a memorable analogy, a concrete recommendation with one specific reason, and a named anti-pattern.
</details>

### Stretch exercise D — ⭐⭐⭐ Visualise attention (conceptually)

For the sentence *"The animal didn't cross the street because **it** was too tired."*, the pronoun "it" is ambiguous — it could refer to "the animal" or "the street".

1. Which one does it refer to? How do you know?
2. In your own words, describe what the model's attention weights for the word "it" would probably look like across the rest of the sentence — which words would have the highest weight, and why?
3. Now do the same for *"...because **it** was too wide."* — what changes?

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**1. Refers to "the animal".** You know this because *"tired"* is a property animals have, not streets.

**2. Attention weights for "it":** highest on **"animal"** (the antecedent it resolves to), with secondary weight on **"tired"** (the property that determines the resolution). Low weight on "street", "didn't", "cross", because they're irrelevant for resolving "it".

**3. With "too wide":** the resolution flips. "It" now refers to **"the street"** (streets have widths, animals don't usually). Attention weights for "it" would shift to put highest weight on **"street"** and secondary on **"wide"**.

**Why this matters:** this is exactly the kind of context-dependent disambiguation that *requires* attention to see the *whole* sentence at once. A model that processes left-to-right one word at a time would see "it" *before* "tired" or "wide" — and would have to commit to a referent before the disambiguating word arrived. Attention solves this by letting the model wait until all the evidence is in. This famous example is from the original Transformer paper visualisations.
</details>

## 🎁 Bonus mini-project — Build a token-budget calculator

Write a small Streamlit app (or just a Python function) that:

1. Takes a piece of text as input.
2. Uses `tiktoken` to count exact tokens (`pip install tiktoken`).
3. Shows the cost of sending that text as a prompt to three different LLMs at current API prices.
4. Warns if the text exceeds a chosen context-window limit.

This is the kind of small tool that pays for itself in days when you're running LLM API calls in production — and it's a great test of whether you internalised the token concept.


## 🧠 Key takeaways

- LLMs are neural networks with billions of *parameters* that generate text by predicting the next *token*.
- The **Transformer + Attention** architecture is what made modern LLMs possible — it processes context in parallel and captures long-range dependencies.
- Three intervention points: **pre-training** (only the provider does it), **fine-tuning** (change behaviour), **prompting** (steer the existing model).
- Five prompting techniques to know: zero-shot, few-shot, chain-of-thought, role assignment, format constraint.
- Two structural limitations: **hallucinations** (fabricated content) and **knowledge cutoff** (no fresh data, no private data). Both are addressable with RAG (NB 29) and tools (NB 30).

> 🧭 **The whole notebook in one breath.** *An LLM is a next-token predictor run in a loop* (sample → append → repeat). Tokens are what it reads and pays for; parameters are what it knows; attention is how it weighs the context inside each loop step; prompting is how you shape that context; and its two failure modes — hallucination and knowledge cutoff — are that same loop running with nothing real to ground it. Keep that one sentence and you can re-derive almost everything else.

> 🏢 **And Helpa?** Right now Helpa is a "Just LLM" — fluent, helpful, and dangerously confident about products that don't exist. You've diagnosed *exactly why*. The rest of Module 6 is the cure: NB 22 calls a model from your code, NB 23 grounds it in your real docs (RAG), NB 24 lets it act (tools). Same loop, better context — that's the entire arc.

## ✅ Self-assessment

- I can explain what an LLM does in one sentence to a non-technical stakeholder.
- I can estimate token counts and reason about context-window limits.
- I can sketch the Attention mechanism (Q/K/V → softmax → weighted sum of values).
- I can pick the right prompting technique for a given task.
- I can articulate why hallucinations happen and what RAG fixes.

## 🚀 Next step

→ **NB 22 — AI-Assisted Workflows** (`./22_ai_workflows.ipynb`). Now that you know what an LLM *is*, let's call one from your own code — prompts, structured output, and classification. Helpa's first real lines of code are waiting there.
